In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('/Users/rc/workspaces/llm/data/AutoPolicy.pdf')
pages = loader.load_and_split()

In [5]:
pages

[Document(page_content="975916277 N IC94549  INS POLEND   E POLWHITEFONT 6554CWF3PPNFE7KINFS6UCCKBF0007 RPUID               BDF_AA\nPolicy Number: 975916277 \nRaghunadh Chilukamari \nPage of 1 5Form_SCTNID_CTGRY.XX0222A264_POLEND\nAuto Policy Endorsement\nYour policy is amended as follows:\nPart IV - Damage To A Vehicle\nThe following exclusion is added:\nTo a covered auto while being operated by a driver who, at the time of the loss, was not listed on your \ndeclarations page and who was residing in your household as a permanent resident or as a temporary guest.\nThis exclusion does not apply to the following situations:\na. The driver operating the covered auto started residing in your household as a permanent resident or as a \ntemporary guest no more than 185 days prior to the loss;\nb. The driver operating the covered auto became a licensed driver no more than 185 days prior to the loss. For \npurposes of this exclusion, a licensed driver, includes a driver with an instructional o

In [ ]:
# loader = PyPDFLoader("/Users/rc/workspaces/llm/data/policy-sample-1-15.pdf", extract_images=True)
# pages = loader.load()
# pages[5].page_content

In [10]:
from langchain_community.document_loaders import PDFMinerPDFasHTMLLoader
loader = PDFMinerPDFasHTMLLoader("/Users/rc/workspaces/llm/data/policy-1.pdf")
data = loader.load()[0] 

from bs4 import BeautifulSoup
soup = BeautifulSoup(data.page_content,'html.parser')
content = soup.find_all('div')
soup

<html><head>
<meta content="text/html" http-equiv="Content-Type"/>
</head><body>
<span style="position:absolute; border: gray 1px solid; left:0px; top:50px; width:612px; height:792px;"></span>
<div style="position:absolute; top:50px;"><a name="1">Page 1</a></div>
<div style="position:absolute; border: textbox 1px solid; writing-mode:lr-tb; left:108px; top:133px; width:434px; height:44px;"><span style="font-family: TimesNewRomanPSMT; font-size:9px">Please note that hotel cancellation policies vary. Please avoid booking any hotel with a pre-paid rate as these 
<br/>are non-cancelable and non-refundable without penalty. These hotel rates have been blocked from booking 
<br/>on Concur. If the reservation and/or cancellation is made directly with the hotel, employees should request 
<br/>and retain a reservation and/or cancellation number as documentation of the transaction. 
<br/></span></div><div style="position:absolute; border: textbox 1px solid; writing-mode:lr-tb; left:90px; top:191px

In [2]:

import re
cur_fs = None
cur_text = ''
snippets = []   # first collect all snippets that have the same font size
for c in content:
    sp = c.find('span')
    if not sp:
        continue
    st = sp.get('style')
    if not st:
        continue
    fs = re.findall('font-size:(\d+)px',st)
    if not fs:
        continue
    fs = int(fs[0])
    if not cur_fs:
        cur_fs = fs
    if fs == cur_fs:
        cur_text += c.text
    else:
        snippets.append((cur_text,cur_fs))
        cur_fs = fs
        cur_text = c.text
snippets.append((cur_text,cur_fs))
# Note: The above logic is very straightforward. One can also add more strategies such as removing duplicate snippets (as
# headers/footers in a PDF appear on multiple pages so if we find duplicates it's safe to assume that it is redundant info)

In [3]:
from langchain.docstore.document import Document
cur_idx = -1
semantic_snippets = []
# Assumption: headings have higher font size than their respective content
for s in snippets:
    # if current snippet's font size > previous section's heading => it is a new heading
    if not semantic_snippets or s[1] > semantic_snippets[cur_idx].metadata['heading_font']:
        metadata={'heading':s[0], 'content_font': 0, 'heading_font': s[1]}
        metadata.update(data.metadata)
        semantic_snippets.append(Document(page_content='',metadata=metadata))
        cur_idx += 1
        continue

    # if current snippet's font size <= previous section's content => content belongs to the same section (one can also create
    # a tree like structure for sub sections if needed but that may require some more thinking and may be data specific)
    if not semantic_snippets[cur_idx].metadata['content_font'] or s[1] <= semantic_snippets[cur_idx].metadata['content_font']:
        semantic_snippets[cur_idx].page_content += s[0]
        semantic_snippets[cur_idx].metadata['content_font'] = max(s[1], semantic_snippets[cur_idx].metadata['content_font'])
        continue

    # if current snippet's font size > previous section's content but less than previous section's heading than also make a new
    # section (e.g. title of a PDF will have the highest font size but we don't want it to subsume all sections)
    metadata={'heading':s[0], 'content_font': 0, 'heading_font': s[1]}
    metadata.update(data.metadata)
    semantic_snippets.append(Document(page_content='',metadata=metadata))
    cur_idx += 1

In [5]:
semantic_snippets[0]

Document(page_content='(In many European countries, tipping is already included and \notherwise minimal) \n', metadata={'heading': 'Please note that hotel cancellation policies vary. Please avoid booking any hotel with a pre-paid rate as these \nare non-cancelable and non-refundable without penalty. These hotel rates have been blocked from booking \non Concur. If the reservation and/or cancellation is made directly with the hotel, employees should request \nand retain a reservation and/or cancellation number as documentation of the transaction. \n12. Rerouting/Changing Travel Plans \nWhen  rerouting  or  other  charges  are  required  while  on  a  trip,  the  employee  must  contact  Gant  Travel \nManagement to request the  change. If the change  is being made  more than 24 hours prior to the  change, \nplease email Gant at travelsupport@ganttravel.com. If the change is within 24 hours, the traveler must call a \nGant Travel agent at 877-924-0303 to make the necessary change. If phon

In [3]:
from llmsherpa.readers import LayoutPDFReader

llmsherpa_api_url = "https://readers.llmsherpa.com/api/document/developer/parseDocument?renderFormat=all"
pdf_url = "/Users/rc/workspaces/llm/data/policy-1.pdf" # also allowed is a file path e.g. /home/downloads/xyz.pdf
pdf_reader = LayoutPDFReader(llmsherpa_api_url)
doc = pdf_reader.read_pdf(pdf_url)

In [5]:
for chunk in doc.chunks():
    print(chunk.to_text())

Please note that hotel cancellation policies vary.
Please avoid booking any hotel with a pre-paid rate as these are non-cancelable and non-refundable without penalty.
These hotel rates have been blocked from booking on Concur.
If the reservation and/or cancellation is made directly with the hotel, employees should request and retain a reservation and/or cancellation number as documentation of the transaction.
When rerouting or other charges are required while on a trip, the employee must contact Gant Travel Management to request the change.
If the change is being made more than 24 hours prior to the change, please email Gant at travelsupport@ganttravel.com.
If the change is within 24 hours, the traveler must call a Gant Travel agent at 877-924-0303 to make the necessary change.
If phone or internet access is restricted, contact the Global Travel Coordinator for assistance.
Any additional expense or reduction in fare resulting from the change must be accounted for by the employee on the

In [6]:
for section in doc.sections():
    print(section.title)

12. Rerouting/Changing Travel Plans
13. Tipping Guidelines
14. Other Reimbursable Expenses
15. Per Diem vs Company Expense/Credit Cards


In [8]:
for table in doc.tables():
    print(table.to_text())

 | Purpose | If per diem is used | If corporate credit card is used
 | --- | --- | ---
 | Meals | Covered in meal per diem rate | 15% full service 5% buffet/pick-up/delivery
 | (In many European countries, tipping is already included and otherwise minimal)
 | Hired Car (Uber, Taxi, etc.) | N/A | 10%
 | Bell hop, shuttle bus driver, etc. | Covered in incidental | 



In [11]:
from IPython.core.display import display, HTML

def get_section_text(doc, section_title):
    """
    Extracts the text from a specific section in a parsed PDF document.

    Parameters:
    - doc (Document): A Document object from the llmsherpa.readers.layout_reader library.
    - section_title (str): The title of the section to extract.

    Returns:
    - str: The HTML representation of the section's content, or a message if the section is not found.
    """

    selected_section = None

    # Find the desired section by title
    for section in doc.sections():
        if section.title == section_title:
            selected_section = section
            break

    # If the section is not found, return a message
    if not selected_section:
        return f"No section titled '{section_title}' found."

    # Return the full content of the section as HTML
    return selected_section.to_html(include_children=True, recurse=True)

/var/folders/2w/0kqpwndd7394lk_g4l7487jc0000gn/T/ipykernel_2521/3030755453.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [16]:
section_text = get_section_text(doc, '15. Per Diem vs Company Expense/Credit Cards')
HTML(section_text)